# Akıllı Sınıf - İki Aşamalı AI Ajanı Eğitimi (Sensör Füzyonu)
Bu notebook'ta Radar ve CO2 verilerini birleştirerek sınıfın doluluğunu tahmin eden, ardından bu tahmin ve sıcaklığa bakarak Klima Gücüne (0-3) karar veren Uçtan Uca (End-to-End) bir model eğiteceğiz.

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, Concatenate
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import json

print("Kütüphaneler yüklendi. TensorFlow Versiyonu:", tf.__version__)

## 1. Veri Setinin Yüklenmesi ve Hazırlanması
Radar, CO2, Sıcaklık verilerini alacağız.

In [ ]:
df = pd.read_csv("../2_Veri_Toplama/dataset.csv")

radar_cols = [col for col in df.columns if "Move" in col or "Stat" in col]
X = df[radar_cols + ["CO2_Egimi", "Sicaklik", "Nem"]].values

y_kisi = df["Gercek_Kisi_Sayisi"].values
y_klima = df["Hedef_Klima_Gucu"].values

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Scaler değerlerini kaydet (Arayüzde gerçek tahmin için)
with open('scaler.json', 'w') as f:
    json.dump({"min": scaler.data_min_.tolist(), "scale": scaler.scale_.tolist()}, f)

X_train, X_test, y_kisi_train, y_kisi_test, y_klima_train, y_klima_test = train_test_split(
    X_scaled, y_kisi, y_klima, test_size=0.2, random_state=42
)

print(f"Eğitim verisi: {X_train.shape[0]} satır. Test verisi: {X_test.shape[0]} satır.")

## 2. İki Aşamalı Yapay Zeka Mimarisinin Kurulması

In [ ]:
inputs = Input(shape=(X_train.shape[1],), name="sensor_inputs")

x1 = Dense(32, activation='relu')(inputs)
x1 = Dropout(0.2)(x1)
x1 = Dense(16, activation='relu')(x1)
kisi_ciktisi = Dense(1, activation='relu', name="kisi_tahmini")(x1)

sicaklik_nem = inputs[:, -2:] 
birlestirilmis = Concatenate()([kisi_ciktisi, sicaklik_nem])

x2 = Dense(16, activation='relu')(birlestirilmis)
klima_ciktisi = Dense(4, activation='softmax', name="klima_karari")(x2)

model = Model(inputs=inputs, outputs=[kisi_ciktisi, klima_ciktisi])

model.compile(
    optimizer='adam',
    loss={'kisi_tahmini': 'mse', 'klima_karari': 'sparse_categorical_crossentropy'},
    loss_weights={'kisi_tahmini': 1.0, 'klima_karari': 2.0},
    metrics={'klima_karari': 'accuracy'}
)

model.summary()

## 3. Modelin Eğitilmesi

In [ ]:
history = model.fit(
    X_train, 
    {'kisi_tahmini': y_kisi_train, 'klima_karari': y_klima_train},
    validation_data=(X_test, {'kisi_tahmini': y_kisi_test, 'klima_karari': y_klima_test}),
    epochs=50,
    batch_size=32,
    verbose=1
)

## 4. Eğitimi İnceleme ve TFLite Dönüşümü

In [ ]:
plt.plot(history.history['klima_karari_accuracy'], label='Eğitim Başarısı')
plt.plot(history.history['val_klima_karari_accuracy'], label='Doğrulama Başarısı')
plt.title('Klima Kararı AI Ajanı Başarısı')
plt.legend()
plt.show()

# Hem Keras hem TFLite olarak kaydet (Keras'ı Streamlit kullanacak)
model.save('ai_ajani.keras')

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open('ai_ajani.tflite', 'wb') as f:
    f.write(tflite_model)
    
print("Modeller kaydedildi!")